In [ ]:
from IPython.display import HTML, display

display(HTML("""
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:wght@400;500;600&family=Instrument+Serif:ital@0;1&display=swap" rel="stylesheet">
<style>
  .pg-hero {
    --ink: #14212b;
    --muted: #5a6b78;
    --line: #cfd8e0;
    --panel: #fbfcfd;
    --accent: #0f5c4c;
    --accent-soft: #e4f1ed;
    font-family: "IBM Plex Sans", system-ui, sans-serif;
    color: var(--ink);
    max-width: 42rem;
    margin: 0 auto 1.25rem;
    padding: 1.75rem 1.5rem 1.5rem;
    background:
      linear-gradient(160deg, #e7edf2 0%, #f2f5f7 48%, #e8efeb 100%);
    border: 1px solid var(--line);
    border-radius: 2px;
    box-sizing: border-box;
  }
  .pg-hero * { box-sizing: border-box; }
  .pg-hero .brand {
    font-family: "Instrument Serif", Georgia, serif;
    font-size: 2.15rem;
    font-weight: 400;
    line-height: 1.15;
    letter-spacing: -0.01em;
    margin: 0 0 0.55rem;
  }
  .pg-hero .job {
    font-size: 1.05rem;
    line-height: 1.45;
    color: var(--muted);
    margin: 0 0 1rem;
    max-width: 34rem;
  }
  .pg-hero .disclaimer {
    display: inline-block;
    font-size: 0.78rem;
    font-weight: 500;
    letter-spacing: 0.02em;
    text-transform: uppercase;
    color: var(--accent);
    background: var(--accent-soft);
    border: 1px solid #b7d5cc;
    padding: 0.28rem 0.55rem;
    margin: 0 0 1.35rem;
  }
  .pg-hero h2 {
    font-family: "IBM Plex Sans", system-ui, sans-serif;
    font-size: 0.72rem;
    font-weight: 600;
    letter-spacing: 0.08em;
    text-transform: uppercase;
    color: var(--muted);
    margin: 0 0 0.65rem;
  }
  .pg-hero ol {
    margin: 0 0 1.35rem;
    padding: 0;
    list-style: none;
    counter-reset: step;
  }
  .pg-hero ol li {
    counter-increment: step;
    position: relative;
    padding: 0.35rem 0 0.35rem 2rem;
    font-size: 0.95rem;
    line-height: 1.4;
  }
  .pg-hero ol li::before {
    content: counter(step);
    position: absolute;
    left: 0;
    top: 0.28rem;
    width: 1.35rem;
    height: 1.35rem;
    border-radius: 50%;
    background: var(--panel);
    border: 1px solid var(--line);
    color: var(--accent);
    font-size: 0.72rem;
    font-weight: 600;
    display: flex;
    align-items: center;
    justify-content: center;
  }
  .pg-hero .formats {
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 0.55rem;
    margin: 0 0 1.15rem;
  }
  .pg-hero .fmt {
    background: var(--panel);
    border: 1px solid var(--line);
    padding: 0.7rem 0.75rem;
  }
  .pg-hero .fmt strong {
    display: block;
    font-size: 0.88rem;
    font-weight: 600;
    margin-bottom: 0.15rem;
  }
  .pg-hero .fmt span {
    font-size: 0.78rem;
    color: var(--muted);
    line-height: 1.35;
  }
  .pg-hero .note {
    font-size: 0.82rem;
    line-height: 1.45;
    color: var(--muted);
    margin: 0;
    padding-top: 0.85rem;
    border-top: 1px solid var(--line);
  }
  .pg-hero a { color: var(--accent); }
  @media (max-width: 560px) {
    .pg-hero .formats { grid-template-columns: 1fr; }
    .pg-hero .brand { font-size: 1.85rem; }
  }
</style>
<section class="pg-hero" aria-label="Paul Graham essay feeds">
  <p class="brand">Paul Graham Essay Feeds</p>
  <p class="job">Generate unofficial RSS, Atom, and JSON Feed files for
    <a href="https://paulgraham.com/articles.html" target="_blank" rel="noopener">paulgraham.com/articles.html</a>
    — then download a zip for your reader.</p>
  <p class="disclaimer">Unofficial · not affiliated with Paul Graham</p>

  <h2>How to use</h2>
  <ol>
    <li>Runtime → <strong>Run all</strong></li>
    <li>Wait while feeds are built — enrich may take several minutes; the run also live-checks essay URLs</li>
    <li>Download <strong>feeds.zip</strong> when prompted (link issues show in a status panel, still downloadable)</li>
  </ol>

  <h2>What you get</h2>
  <div class="formats">
    <div class="fmt"><strong>RSS 2.0</strong><span>rss.xml</span></div>
    <div class="fmt"><strong>Atom 1.0</strong><span>atom.xml</span></div>
    <div class="fmt"><strong>JSON Feed 1.1</strong><span>feed.json</span></div>
  </div>

  <p class="note">Metadata only — titles, links, short source-derived summaries.
    No full essay bodies. A durable <code>catalog.json</code> is written under the output root;
    the zip contains only the three feeds. Installs from the repo <code>main</code> branch via <code>uvx</code>.</p>
</section>
"""))

In [ ]:
#@title Generate feeds
#@markdown Leave **Enrich** on for short per-essay summaries (~1 HTTP GET each).
#@markdown Turn it off for a faster index-only run.
#@markdown Live link checks stay on (report-only); issues appear in the status panel below.
ENRICH = True  #@param {type:"boolean"}
#@markdown ---
#@markdown ### Advanced
ROOT = "/content/pg-feeds"  #@param {type:"string"}

!pip install -q "uv>=0.12"

import html
import re
import subprocess
import zipfile
from pathlib import Path

from IPython.display import HTML, display

root = Path(ROOT).expanduser().resolve()
root.mkdir(parents=True, exist_ok=True)
pkg = "git+https://github.com/wyattowalsh/paul-graham-essay-feeds@main"


def run_uvx(argv: list[str]) -> str:
    """Run pg-essay-feeds via uvx; print logs; return combined output."""
    cmd = ["uvx", "--from", pkg, "pg-essay-feeds", *argv]
    proc = subprocess.run(cmd, text=True, capture_output=True, check=False)
    chunks: list[str] = []
    if proc.stdout:
        print(proc.stdout, end="" if proc.stdout.endswith("\n") else "\n")
        chunks.append(proc.stdout)
    if proc.stderr:
        print(proc.stderr, end="" if proc.stderr.endswith("\n") else "\n")
        chunks.append(proc.stderr)
    if proc.returncode != 0:
        raise subprocess.CalledProcessError(
            proc.returncode, cmd, proc.stdout, proc.stderr
        )
    return "".join(chunks)


def parse_link_probe_report(log: str) -> tuple[bool, int, list[str]]:
    """Parse enrich.py probe lines. Returns (ok, failure_count, messages)."""
    issues: list[str] = []
    for line in log.splitlines():
        marker = "Link probe issue:"
        if marker in line:
            msg = line.split(marker, 1)[1].strip()
            if msg:
                issues.append(msg)
    count_match = re.search(r"(\d+) link probe failure\(s\)", log)
    if count_match is not None:
        count = int(count_match.group(1))
        return False, count, issues[:10]
    if issues:
        return False, len(issues), issues[:10]
    return True, 0, []


update_argv = ["update", "--repo-root", str(root)]
if not ENRICH:
    update_argv.append("--no-enrich")
# validate_links stays on (package default True) — do not pass --no-validate-links

update_log = run_uvx(update_argv)
run_uvx(["check", "--repo-root", str(root)])

feeds = root / "feeds"
names = ("rss.xml", "atom.xml", "feed.json")
missing = [n for n in names if not (feeds / n).is_file()]
if missing:
    raise FileNotFoundError(
        f"Expected feed files under {feeds}, missing: {', '.join(missing)}"
    )

zip_path = root / "feeds.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in names:
        zf.write(feeds / name, arcname=name)

try:
    from google.colab import files

    files.download(str(zip_path))
    where = "Your browser should download <strong>feeds.zip</strong> now."
except Exception:
    where = f"Open the zip at <code>{html.escape(str(zip_path))}</code>."

probes_ok, probe_count, probe_msgs = parse_link_probe_report(update_log)
catalog_note = (
    f"Durable <code>catalog.json</code> is on disk under <code>{html.escape(str(root))}</code>; "
    "the zip is feeds-only."
)

if probes_ok:
    display(HTML(f"""
<div style="
  font-family: 'IBM Plex Sans', system-ui, sans-serif;
  max-width: 42rem; margin: 1rem auto 0; padding: 1.1rem 1.25rem;
  background: #e4f1ed; border: 1px solid #b7d5cc; color: #14212b;
">
  <p style="
    font-family: 'Instrument Serif', Georgia, serif;
    font-size: 1.35rem; margin: 0 0 0.35rem; color: #0f5c4c;
  ">Your feeds are ready</p>
  <p style="margin: 0 0 0.45rem; font-size: 0.92rem; line-height: 1.45; color: #5a6b78;">
    {where} Contains <code>rss.xml</code>, <code>atom.xml</code>, and <code>feed.json</code>.
  </p>
  <p style="margin: 0; font-size: 0.82rem; line-height: 1.4; color: #5a6b78;">
    Live link probes OK. {catalog_note}
  </p>
</div>
"""))
else:
    items = "".join(
        f"<li style='margin: 0.2rem 0;'><code>{html.escape(m)}</code></li>"
        for m in probe_msgs
    )
    more = (
        f"<p style='margin: 0.45rem 0 0; font-size: 0.78rem; color: #7a5a2b;'>"
        f"Showing {len(probe_msgs)} of {probe_count}.</p>"
        if probe_count > len(probe_msgs)
        else ""
    )
    display(HTML(f"""
<div style="
  font-family: 'IBM Plex Sans', system-ui, sans-serif;
  max-width: 42rem; margin: 1rem auto 0; padding: 1.1rem 1.25rem;
  background: #f7f0e4; border: 1px solid #e0c99a; color: #14212b;
">
  <p style="
    font-family: 'Instrument Serif', Georgia, serif;
    font-size: 1.35rem; margin: 0 0 0.35rem; color: #8a5a12;
  ">Feeds ready — {probe_count} link probe issue(s)</p>
  <p style="margin: 0 0 0.55rem; font-size: 0.92rem; line-height: 1.45; color: #5a6b78;">
    {where} Essays are still included; <strong>feeds.zip</strong> is ready.
    Contains <code>rss.xml</code>, <code>atom.xml</code>, and <code>feed.json</code>.
  </p>
  <p style="margin: 0 0 0.35rem; font-size: 0.82rem; font-weight: 600; color: #7a5a2b;">
    Report-only live link checks (does not block the zip):
  </p>
  <ul style="margin: 0; padding-left: 1.2rem; font-size: 0.82rem; line-height: 1.4; color: #5a6b78;">
    {items}
  </ul>
  {more}
  <p style="margin: 0.65rem 0 0; font-size: 0.82rem; line-height: 1.4; color: #5a6b78;">
    {catalog_note}
  </p>
</div>
"""))

<details>
<summary><strong>Troubleshooting</strong> (optional)</summary>

- **No download dialog** — re-run the generate cell, or look for `feeds.zip` under the path in Advanced (`/content/pg-feeds` by default).
- **Update failed / network error** — Runtime → Run all again; Colab must reach GitHub and paulgraham.com.
- **Slow run** — Enrich fetches each essay page; live link probes also hit every URL. Both are on by default.
- **Amber “link probe issue(s)” panel** — report-only; essays stay in the feeds and `feeds.zip` still downloads. Scroll shell output for full `Link probe issue:` lines.
- **Want it faster** — uncheck Enrich (index titles/links only, no per-page summaries). Link probes still run.
- **Where is catalog.json?** — written under Advanced `ROOT` (durable SSOT). The zip is feeds-only (`rss.xml`, `atom.xml`, `feed.json`).
- **Local instead** — see the repo [README](https://github.com/wyattowalsh/paul-graham-essay-feeds#readme) for `uvx` without Colab.

</details>